# Data Visualisation and Data Preprocessing for Housing Dataset

CONTENT 
1. Data Loading
2. Data Analysis
3. Data Visualization
4. Train Test Split with Random Sampling
5. Train Test Split with Stratified Sampling
6. Correlations
----------------
7. Feature Extraction
8. Data Conversion from Text to Numeric 
9. Data Cleaning 

PREREQUISITES 
1. Jupyter Notebook Working with Python 3+ 
2. Check python version in top right corner OR Execute the following 


In [ ]:
import numpy as np

In [ ]:
import sys
sys.version

In [ ]:
import urllib

In [ ]:
import os                          # OS       Standard file operations at OS level
import tarfile                     # TARFILE  Operations related to compressi import urllib       # URLLIB   Operations on web pages  
import pandas as pd                # PANDAS   Data Manipulation and Data Analysis


#### Data Loading

Local location for data storage

In [ ]:
# Variables - Usually you store these variables in config files 
HOUSING_WEB_PATH = "https://raw.githubusercontent.com/ageron/handson-ml/master/datasets/housing/housing.tgz"
HOUSING_LOCAL_FOLDER = "datasets/housing"
HOUSING_DATA_FILE = HOUSING_LOCAL_FOLDER + "/housing.tgz"

Data Fetching from a URL 

In [ ]:
# Variables - Usually initialised in the source code  
source = HOUSING_WEB_PATH
destination = HOUSING_DATA_FILE
destination_folder = HOUSING_LOCAL_FOLDER

In [ ]:
urllib.request.urlretrieve(source, destination)

In [ ]:
urllib.request.urlretrieve(source, destination)

In [ ]:
#Create the folder first 
if not os.path.isdir(HOUSING_LOCAL_FOLDER):
    os.makedirs(HOUSING_LOCAL_FOLDER)

In [ ]:
housing_tgz.close()

Modularize the Code

In [ ]:
# FETCH the data from a web link
# EXTRACT the files into a local storage
def fetch_housing_data(source_url=HOUSING_WEB_PATH, 
                       housing_path=HOUSING_LOCAL_FOLDER):
    if source_url is None:
        # In case it is already downloaded 
        def load_housing_data(housing_path):
            csv_path = os.path.join(housing_path, "housing.csv")
            return pd.read_csv(csv_path)
    else:
        
        if not os.path.isdir(housing_path):
            os.makedirs(housing_path)
        tgz_path = os.path.join(housing_path, "housing.tgz")
        urllib.request.urlretrieve(source_url, tgz_path)
        housing_tgz = tarfile.open(tgz_path)
        housing_tgz.extractall(path=housing_path)
        housing_tgz.close()
        
    return load_housing_data(housing_path)
housing = fetch_housing_data(source_url=None, housing_path=HOUSING_LOCAL_FOLDER)

#### Data Loading Completed 

#### Data Analysis 

In [ ]:
# Check housing schema etc with single command
housing.info()

In [ ]:
# Check housing few sample rows in the housing dataframe 
housing.head()

In [ ]:
# Check the summary statistics of the numerical columns in housing dataframe
housing.describe()

In [ ]:
# Set the housing dataframe index as ocean_proximity
df=housing.set_index("ocean_proximity")

In [ ]:
df.head()

###### Difference between .loc and .iloc operations  

In [ ]:
df.loc["NEAR BAY"].head()

In [ ]:
df.iloc[0:5] # JUST LIKE df.head(n=5)

In [ ]:
df.loc[0:3]

In [ ]:
#Notes
# .loc  => works with exact index values
# .iloc => works when you want values/rows from partaicular range of numbers
# One more thing to note between .loc and .iloc is:
# .loc[low:high] is inclusive of high 
# .iloc[low:high] is exclusive of high and run till high - 1 

In [ ]:
housing.loc[0:5]

##### Column by Column Analysis - Eg. Analyse Numerical and categorical columns

In [ ]:
housing.ocean_proximity.value_counts()


In [ ]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
housing_grouped.mean()

In [ ]:
housing_grouped.count()

In [ ]:
def iqr(x):
    q1=np.quantile([x,0.25])
    q3=np.quantile([x,0.75])
    iqr=q3-q1
    return iqr

housing.groupby(["ocean_proximity"]).agg(iqr)

### DATA VISUALISATION - Part 1


In [ ]:
%matplotlib inline 
import matplotlib.pyplot as plt


In [ ]:
housing.plot(kind="hist",subplots=True)

#### Train Test Split 

In [ ]:
def split_train_test(data, test_ratio):
    shuffled_indices = np.random.permutation(len(data))   #Get random indices from 1 to n
    test_set_size = int(len(data) * test_ratio)           # Standard followed is 20% of entire dataset is test set
    test_indices = shuffled_indices[:test_set_size]         
    train_indices = shuffled_indices[test_set_size:]      
    return data.iloc[train_indices], data.iloc[test_indices] 


In [ ]:
train_set, test_set = split_train_test(housing, 0.2)
print(len(train_set), "train +", len(test_set), " test")


In [ ]:
# 2 possible solutions 
# a. save the train and test set 
# b. select test set with some hash function on an identifier 

In [ ]:
import hashlib 
def test_set_check(identifier, test_ratio, hash):
    return hash(np.int64(identifier)).digest()[-1] < 256 * test_ratio

def split_train_test_by_id(data, test_ratio, id_col, hash=hashlib.md5):
    ids = data[id_col]
    in_test_set = ids.apply(lambda id_: test_set_check(id_, test_ratio, hash))
    return data.loc[~in_test_set], data.loc[in_test_set]

housing_with_id = housing.reset_index()
train_set, test_set = split_train_test_by_id(housing_with_id,0.2, "index")

In [ ]:
# Assume median income is the most important attribute to predict median housing prices 
# We need to make sure test is representative of various categories of income
housing.describe()

In [ ]:
housing["income_cat"] = np.ceil(housing["median_income"] / 1.5)            # Divide into categories
housing["income_cat"].where(housing["income_cat"] < 5, 5.0 , inplace=True) 
# Take income less than 5 as it is and label anything above 5  as 5

In [ ]:
# Single command to do the above cut 
housing["income_cat"] = pd.cut(housing["median_income"],
                               bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
                               labels=[1, 2, 3, 4, 5])

In [ ]:
housing["income_cat"].plot(kind="hist",title="Income Categories")

In [ ]:
# Library => Module => function
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.model_selection import train_test_split
import sklearn.model_selection as sample


#ANOTHER WAY
# sample.StratifiedShuffleSplit(...)


# WHY ???
# HOW? ??? (NOT the CODE but strategy)
https://machinelearningaptitude.com/topics/machine-learning/what-is-stratified-sampling-and-why-is-it-important/
https://machinelearningaptitude.com/topics/machine-learning/what-is-the-problem-in-random-or-uniform-sampling-of-test-set-from-the-entire-dataset/

In [ ]:
# https://www.numpy.org/devdocs/user/quickstart.html

STRATIFIED SAMPLING 


In [ ]:
#VALIDATION - k folds , if k = 5, 
object_declared = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in object_declared.split(housing, housing["income_cat"]):
    strat_train_set = housing.loc[train_index]
    strat_test_set = housing.loc[test_index]
    

CHECK if this sampling worked

In [ ]:
# Entire dataset 
housing["income_cat"].value_counts() / len(housing)

In [ ]:
housing["income_cat"].value_counts(normalize=True)

In [ ]:
test_set["income_cat"].value_counts()/len(strat_test_set)

In [ ]:
# TEST SET 
strat_test_set["income_cat"].value_counts()/len(strat_test_set)

Drop the extra column 

In [ ]:
# Drop income_cat
# for df in (strat_train_set, strat_test_set):
for df in (train_set, test_set):
    df.drop(["income_cat"], axis=1, inplace=True)

In [ ]:
# housing = strat_train_set.copy()
housing = train_set.copy()
# NOTICE why is it copy() and not just direct assignment

In [ ]:
# NOTE THAT test set was created before 
# WE ALWAYS DO TRAIN _ TEST SPLIT BEFORE PRE PROCESSING!! 

DATA VISUALIZATION - Part 2

In [ ]:
# Scatter plot
# X and Y should be in the dataframe

In [ ]:
housing.drop(["median_house_value"],axis=1).plot()

In [ ]:
housing[["population","total_bedrooms"]].plot(kind="hist")

In [ ]:
# Plot mulitple things for more understanding 
# x,y- Longitude and latitude on x and y axis resp.  (2 features)
# alpha- High density zones
# s - Bubbles radius proportional to area population (1 feature) (s-SIZE of bubble)
# c- color represents the house price 

In [ ]:
housing.plot(kind="scatter", x="longitude", y="latitude",alpha=1,
            s=housing["population"]/100, label="population", 
            c="median_house_value", cmap=plt.get_cmap("jet"), colorbar=True,figsize=(12,8))
plt.legend()


CORRELATIONS 

In [ ]:
import seaborn as sns
sns.heatmap(corr_mtx)

In [ ]:
housing[(housing["ocean_proximity"]=="ISLAND") | 
        (housing["ocean_proximity"]=="INLAND")].corr()

In [ ]:
corr_mtx["median_house_value"].sort_values(ascending=False)

In [ ]:
from pandas.plotting import scatter_matrix
att = ["median_house_value","median_income", "total_rooms", "housing_median_age" ]

In [ ]:
scatter_matrix(housing[att], figsize=(12,8))

In [ ]:
housing.plot(kind="scatter", x="median_income", y="median_house_value", alpha=0.1)

Remember - we played with linear correlations

Correlation might be close to 0 but attributes might still have non linear relationship

Till now - Correlations, visualisation

### FEATURE EXTRACTION 

In [ ]:
# Difference between feature extraction and feature selection
# Use existing features or the data, subject expertise to derive new features
# Select relevant features for the problem

housing.head()
# From data analysis and data visualisation - 
# derive some features and check how are things then

Re calculate the correlation for new features 

In [ ]:
corr_matrix["median_house_value"].sort_values(ascending=False)

In [ ]:
# Interestingly, bedrooms_per_room has high correlation with median_house_value than some original features
# total_rooms                  0.135097
# total_bedrooms               0.047689  

# bedrooms_per_room           -0.259984

In [ ]:
# DROP true label from the training set, was not removed till now for the analysis 
housing = train_set.drop(["median_house_value"], axis=1)
housing_labels = train_set["median_house_value"].copy()

### DATA CLEANING - ADD MISSING VALUES 

Scikit Learn's class for handling missing values - Imputer

In [ ]:
# from sklearn.preprocessing import Imputer
# imputer = Imputer(strategy="median")
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="median")


In [ ]:
# Median can be computed only on numerical data - remove non numeric data
housing_num = housing.drop(["ocean_proximity"],axis=1,inplace=False)

In [ ]:
imputer.fit(housing_num)

In [ ]:
print(imputer.statistics_)
print(housing_num.median().values)

In [ ]:
X = imputer.transform(housing_num)
# X = imputer.fit_transform(housing_num)

### Handling TEXT and CATEGORICAL Data 

In [ ]:
housing.ocean_proximity.value_counts()

In [ ]:
# OLDER VERSIONS - LabelEncoder convert each cateogary to a unique integer 
from sklearn.preprocessing import LabelEncoder
en = LabelEncoder()
housing_cat = housing["ocean_proximity"]
housing_cat_encoded = en.fit_transform(housing_cat)
housing_cat_encoded

In [ ]:
# Now use OrdinalEncoder 
from sklearn.preprocessing import OrdinalEncoder
en = OrdinalEncoder()
housing_cat = housing["ocean_proximity"]
housing_cat_encoded = en.fit_transform(housing_cat)
housing_cat_encoded

In [ ]:
print(en.classes_)

ML algorithms will assume that two nearby values are similar than two distant values. For eg., Numerically 0 and 1 are closer but in this context or example, 0(<1H OCEAN) and 4(NEAR OCEAN) are more closer

In [ ]:
# ALSO
# LabelEncoder was made for converting text labels(y) to integers
# We need something specific for the features(X) 

In [ ]:
from sklearn.preprocessing import OneHotEncoder 
en= OneHotEncoder(categories='auto')
# IN PREVIOUS sci-kit learn VERSIONS, OneHotEncoder could only take numerical categories
# NOW, it can take
# housing_cat_1hot = en.fit_transform(housing_cat_encoded.reshape(-1,1))
housing_cat_1hot = en.fit_transform(housing_cat.reshape(-1,1))
housing_cat_1hot

In [ ]:
#LabelBinarizer does both the operations combined

In [ ]:
# NOT REQuiRED ANY MORE as OneHotEncoder is enough
from sklearn.preprocessing import LabelBinarizer
en = LabelBinarizer()
# en = LabelBinarizer(sparse_output=True)  # For sparse matrix 
housing_label_bin = en.fit_transform(housing_cat)
housing_label_bin

In [ ]:
en.classes_

##### REVISE the steps, may not be ordered 

Train-Test Split using Stratified/Random split
Data Visualization - histograms, scatter-plots, boxplots
Data Processing - Text/Categorical Data, Missing Values
Feature Engineering - Extraction, Selection
NEXT WHAT!! 
Combine all the baove steps in a pipeline
Scaling of features
Build Custom Transformers for feature transformation

### Sci-kit  Learn Piplelines

Pipeline helps to combine multiple sequential steps in a single operation
One can have separate pipepines for numerical columns and categorical columns
Both the operations involving categorical and numerical features can execute in parallel
Once both the numerical and categorical pipelines have finished executing, final pipeline with both the operations can be combined into a single operation 

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
rooms_ix, bedrooms_ix, population_ix, household_ix = 3, 4, 5, 6

# get feature index from the data instead of setting hard coded values  
rooms_ix, bedrooms_ix, population_ix, household_ix = [
    list(housing.columns).index(col)
    for col in ("total_rooms", "total_bedrooms", "population", "households")]

class CombinedAttributesAdder(BaseEstimator, TransformerMixin):
    def __init__(self, add_bedrooms_per_room = True): # no *args or **kargs
        self.add_bedrooms_per_room = add_bedrooms_per_room
    def fit(self, X, y=None):
        return self  
    def transform(self, X, y=None):
        rooms_per_household = X[:, rooms_ix] / X[:, household_ix]
        population_per_household = X[:, population_ix] / X[:, household_ix]
        if self.add_bedrooms_per_room:
            bedrooms_per_room = X[:, bedrooms_ix] / X[:, rooms_ix]
            return np.c_[X, rooms_per_household, population_per_household,
                         bedrooms_per_room]
        else:
            return np.c_[X, rooms_per_household, population_per_household]

In [ ]:
housing.head()

##### Numerical pipeline 

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
#2 steps 
mm = MinMaxScaler()
# Fit => done usually from training data
# Transoform  => done on the test data/ upcomin new data 
# ss = StandardScaler(<ARGUMENTS>)
# ss.fit(<TRAINING DATA>)
# ss.transforom(<ANY DATA>)
from sklearn.pipeline import Pipeline

In [ ]:
num_pipeline = Pipeline([
    ('imputer',SimpleImputer(strategy='median')),
    ('attribs_adder', CombinedAttributesAdder()),
    ('std_scaler',StandardScaler()),
])
housing_num_tr = num_pipeline.fit_transform(housing_num)
# num_pipeline.transform(new_sample)

In [ ]:
class MyLabelBinarizer(BaseEstimator, TransformerMixin):
    def __init__(self, *args, **kwargs):
        self.encoder = LabelBinarizer(*args, **kwargs)
        #self.classes_, self.y_type_, self.sparse_input_ = self.encoder.classes_, self.encoder.y_type_, self.encoder.sparse_input_
    def fit(self, x, y=0):
        self.encoder.fit(x)
        return self
    def transform(self, x, y=0):
        return self.encoder.transform(x)

In [ ]:
class DataFrameSelector(BaseEstimator, TransformerMixin):
    def __init__(self, attribute_names):
        self.attribute_names=attribute_names
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X[self.attribute_names].values

In [ ]:
from sklearn.pipeline import FeatureUnion

num_attribs = list(housing_num)

num_pipeline = Pipeline([
    ('selector', DataFrameSelector(num_attribs)),
    ('imputer',Imputer(strategy='median')),
    ('attribs_adder', CombinedAttributesAdder()),
    ('std_scaler',StandardScaler()),
])


##### Categorical pipeline 

In [ ]:
cat_attribs = ["ocean_proximity"]
cat_pipeline = Pipeline([
    ('selector', DataFrameSelector(cat_attribs)),
    ('label_binarizer', MyLabelBinarizer()),
])


##### Combine both the pipelines 

In [ ]:
# OLDER VERSIONS 
full_pipeline = FeatureUnion(transformer_list = [("num_pipeline",num_pipeline),
                                                 ("cat_pipeline",cat_pipeline),])

In [ ]:
from sklearn.compose import ColumnTransformer
full_pipeline = ColumnTransformer([
        ("num", num_pipeline, num_attribs),
        ("cat", OneHotEncoder(), cat_attribs),
    ])

In [ ]:
# https://scikit-learn.org/stable/modules/compose.html#column-transformer

Notice that many data transformation steps are needed and in right order! 

In [ ]:
housing_prepared = full_pipeline.fit_transform(housing)

In [ ]:
housing_prepared.shape

##### Data Preparation is over here 